<a href="https://colab.research.google.com/github/Addreta/Subgraph-Centrality/blob/main/real_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pennylane

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 79.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 22.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 136.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 121.4 MB/s  0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pennylane]


In [ ]:
#this cell holds the whole ideal-circuit pipeline as functions, so the same steps can be run on any graph.
#the steps are: normalization and Chebyshev polynomials (degree fixed by DELTA) -> the two ideal QSVT circuits -> classical SC reference -> ranking comparison.
#bipartite graphs have no odd closed walks, so for them only the even circuit is run (the sinh part is exactly zero).
#the polynomial degree depends only on beta, the normalization alpha and DELTA. No classical score is computed before the circuits are built.
import numpy as np, networkx as nx, pandas as pd, time, warnings
import pennylane as qml
from scipy.linalg import expm
from scipy.special import ive
from scipy.stats import spearmanr, rankdata
from numpy.polynomial import chebyshev as C
warnings.filterwarnings("ignore")

DELTA = 1e-4   # target absolute error on the SC scale; SC(v) >= 1 for unweighted graphs, so it also bounds the relative error. One fixed value for all graphs.

def tie_classes(sc, tol=1e-9):                    # vertices whose SC agree to 1e-9 (relative) are one class
    order = np.argsort(-sc)
    cls = np.zeros(len(sc), dtype=int)
    for a, b in zip(order[:-1], order[1:]):
        cls[b] = cls[a] + int(abs(sc[a] - sc[b]) > tol * abs(sc[a]))
    return cls                                    # class 0 = most central

def cheb_coeffs(z, deg, parity):
    k = np.arange(deg + 1)
    c = 2.0 * ive(k, z)
    c[0] = ive(0, z)
    c[k % 2 != parity] = 0.0
    if parity == 0:
        return c / (0.5 * (1 + np.exp(-2 * z)))    # divide by cosh(z), scaled by e^-z to match ive
    return c / (0.5 * (1 - np.exp(-2 * z)))        # divide by sinh(z), scaled by e^-z to match ive

def tail_error(z, deg, parity):
    return float(np.sum(np.abs(cheb_coeffs(z, deg + 60, parity)[deg + 1:])))

def degree_for_eps(z, eps, parity, max_deg=300):
    d = parity
    while tail_error(z, d, parity) > eps:
        d += 2
        if d > max_deg:
            raise RuntimeError("tolerance not reached; raise max_deg")
    return d, cheb_coeffs(z, d, parity)

def compare(sc_exact, sc_est, k, tol=1e-9):
    # The circuit scores are used as they are. Vertices that are tied in the classical SC may come out in either order.
    ex = np.asarray(sc_exact, dtype=float); est = np.asarray(sc_est, dtype=float); n = len(ex)
    cls = tie_classes(ex, tol)
    # tau: Kendall correlation over the pairs the classical SC orders strictly (= 2 * fraction of correctly ordered pairs - 1)
    iu = np.triu_indices(n, 1)
    dex = (ex[:, None] - ex[None, :])[iu]; dest = (est[:, None] - est[None, :])[iu]
    strict = np.abs(dex) > tol * np.maximum(np.abs(ex[:, None]), np.abs(ex[None, :]))[iu]
    tau = 2 * float(np.mean(np.sign(dex[strict]) == np.sign(dest[strict]))) - 1 if strict.any() else np.nan
    # rho: Spearman correlation of ranks; inside a classical tie class the ranks (classical and estimated) are replaced by their mean rank,
    # so the result does not depend on the order within the class (the scores themselves are not changed)
    r_ex = rankdata(-ex); r_est = rankdata(-est)
    for c in np.unique(cls):
        r_ex[cls == c] = r_ex[cls == c].mean(); r_est[cls == c] = r_est[cls == c].mean()
    rho = spearmanr(r_ex, r_est)[0] if cls.max() > 0 else np.nan
    # top-k: vertices strictly above the k-th classical value must be in the circuit's top-k; ties at the cut count in either order
    t = np.sort(ex)[::-1][k - 1]
    must = set(np.where(ex > t * (1 + tol))[0]); opt = set(np.where(np.abs(ex - t) <= tol * t)[0])
    top = set(np.argsort(-est, kind="stable")[:k])
    overlap = (len(top & must) + min(len(top & opt), k - len(must))) / k
    return dict(rho=rho, tau=tau, overlap=overlap)

def run_graph(G, beta=1.0, weighted=False, delta=None):
    t0 = time.time()
    delta = DELTA if delta is None else delta
    nodes = sorted(G.nodes()); n = len(nodes)
    q = int(np.ceil(np.log2(n))); N = 2 ** q                       # padding: add isolated dummy vertices up to 2^q
    A = nx.to_numpy_array(G, nodelist=nodes, weight="weight" if weighted else None)
    A_p = np.zeros((N, N)); A_p[:n, :n] = A
    bip = nx.is_bipartite(G)
    # ---- normalization and polynomials: the degree comes from beta, alpha and DELTA only ----
    lam = np.max(np.abs(np.linalg.eigvalsh(A)))
    alpha = 1.001 * max(lam, np.sqrt(np.max(np.abs(A @ A).sum(axis=1))))
    z = beta * alpha
    d_e, c_e = degree_for_eps(z, delta / np.cosh(z), 0)
    d_o, c_o = (None, None) if bip else degree_for_eps(z, delta / np.sinh(z), 1)
    # ---- the ideal circuits ----
    At = A_p / alpha
    be_wires = range(q + 1); rp = q + 1
    BE = qml.BlockEncode(At, wires=be_wires)
    dev = qml.device("default.qubit", wires=q + 2)
    def make_circuit(c):
        angles = qml.poly_to_angles(C.cheb2poly(c), "QSVT")
        pos = [qml.PCPhase(a,  dim=N, wires=be_wires) for a in angles]
        neg = [qml.PCPhase(-a, dim=N, wires=be_wires) for a in angles]
        @qml.qnode(dev)
        def circ(v):
            qml.BasisState(np.array([int(b) for b in format(v, f"0{q}b")]), wires=range(1, 1 + q))
            qml.Hadamard(wires=rp)
            qml.ctrl(qml.QSVT(BE, pos), control=rp, control_values=[0])
            qml.ctrl(qml.QSVT(BE, neg), control=rp, control_values=[1])
            qml.Hadamard(wires=rp)
            return qml.probs(wires=range(q + 2))
        return circ
    def p_success(circ, v):
        bits = [0] + [int(b) for b in format(v, f"0{q}b")] + [0]
        return circ(v)[int("".join(map(str, bits)), 2)]
    circ_e = make_circuit(c_e)
    p_e = np.array([p_success(circ_e, v) for v in range(n)])
    if bip:
        p_o = np.zeros(n)
    else:
        circ_o = make_circuit(c_o)
        p_o = np.array([p_success(circ_o, v) for v in range(n)])
    SC_qsvt = np.cosh(z) * np.sqrt(p_e) + np.sinh(z) * np.sqrt(p_o)
    # ---- classical reference (numpy and networkx), used only to check the circuit ----
    SC_exact = np.diag(expm(beta * A))
    nx_diff = np.nan
    if beta == 1.0 and not weighted:                              # networkx's subgraph_centrality is unweighted
        sc_nx = nx.subgraph_centrality(G)
        nx_diff = max(abs(sc_nx[v] - SC_exact[i]) for i, v in enumerate(nodes))
    # ---- comparison ----
    k = 3 if n <= 8 else 5
    m = compare(SC_exact, SC_qsvt, k)
    return dict(n=n, edges=G.number_of_edges(), qubits=q + 2, bipartite=bip, classes=int(tie_classes(SC_exact).max() + 1),
                alpha=alpha, deg_even=d_e, deg_odd=d_o, pe_min=p_e.min(), po_min=(np.nan if bip else p_o.min()),
                nx_diff=nx_diff, MAE=np.mean(np.abs(SC_qsvt - SC_exact)), max_rel_err=np.max(np.abs(SC_qsvt - SC_exact) / SC_exact),
                rho=m["rho"], tau=m["tau"], topk=k, overlap=m["overlap"], seconds=time.time() - t0)

In [ ]:
#this cell lists some of the graphs of the Akrobotu et al. paper (Table 1) and runs the pipeline on every one with beta = 1 (ideal circuits, no noise, no shots).
#BA(50,5,7) is built with seed=7; if the paper used another networkx version its random graph can differ, but it has the same size.
#rho = 1.000 is Spearman’s rank correlation between the exact SC ranking and the QSVT ranking. It’s 1 when the two rankings are identical, 0 when unrelated and -1 when reversed.
#tau = 1.000 is Kendall’s tau. It checks every pair of vertices and asks whether both rankings order the pair the same way. It’s 1 when all pairs agree.
#max rel err is the largest relative error in the SC values themselves, over all vertices: |QSVT value − exact value| / exact value. For example, 1e-5 means the worst vertex is off by 0.001%.
graphs = {
    "Bull":            nx.bull_graph(),
    "K3,5":            nx.complete_bipartite_graph(3, 5),
    "K5":              nx.complete_graph(5),
    "Sedgewick-Maze":  nx.sedgewick_maze_graph(),
    "Florentine":      nx.florentine_families_graph(),
    "Lollipop L10,20": nx.lollipop_graph(10, 20),
    "Davis women":     nx.davis_southern_women_graph(),
    "Karate club":     nx.karate_club_graph(),
    "Tutte":           nx.tutte_graph(),
    "BA(50,5,7)":      nx.barabasi_albert_graph(50, 5, seed=7),

}
rows = {}
for name, G in graphs.items():
    if G is None:
        print(f"{name:<16} skipped (no edge list yet)")
        continue
    rows[name] = run_graph(G, beta=1.0)
    r = rows[name]
    print(f"{name:<16} n = {r['n']:>2}  qubits = {r['qubits']}  rho = {r['rho']:.3f}  tau = {r['tau']:.3f}  max rel err = {r['max_rel_err']:.1e}  ({r['seconds']:.0f} s)")

Bull             n =  5  qubits = 5  rho = 1.000  tau = 1.000  max rel err = 6.1e-06  (1 s)
K3,5             n =  8  qubits = 5  rho = 1.000  tau = 1.000  max rel err = 2.7e-06  (1 s)
K5               n =  5  qubits = 5  rho = nan  tau = nan  max rel err = 1.2e-06  (2 s)
Sedgewick-Maze   n =  8  qubits = 5  rho = 1.000  tau = 1.000  max rel err = 2.2e-05  (2 s)
Florentine       n = 15  qubits = 6  rho = 1.000  tau = 1.000  max rel err = 1.0e-05  (1 s)
Lollipop L10,20  n = 30  qubits = 7  rho = 1.000  tau = 1.000  max rel err = 9.5e-06  (4 s)
Davis women      n = 32  qubits = 7  rho = 1.000  tau = 1.000  max rel err = 1.0e-06  (2 s)
Karate club      n = 34  qubits = 8  rho = 1.000  tau = 1.000  max rel err = 9.3e-07  (17 s)
Tutte            n = 46  qubits = 8  rho = 1.000  tau = 1.000  max rel err = 4.0e-06  (12 s)
BA(50,5,7)       n = 50  qubits = 8  rho = 1.000  tau = 1.000  max rel err = 3.1e-08  (40 s)


In [ ]:
#this cell puts the results of all graphs into one table.
#columns of the results table (one row per graph, ideal circuits, beta = 1):
#n, edges        : number of vertices and edges of the graph
#qubits          : circuit size = register qubits (log2 of the padded size) + 1 block-encoding qubit + 1 real-part qubit
#bipartite       : True if the graph has no odd cycles; then the sinh (odd) part is exactly zero and the odd circuit is skipped
#classes         : number of distinct SC values (vertices with identical SC form one class)
#deg_even/deg_odd: degree of the Chebyshev polynomial for the cosh (even) and sinh (odd) part; the circuit makes one block-encoding call per degree
#pe_min/po_min   : smallest success probability of the even/odd circuit over the vertices (small = many shots needed); po_min is nan for bipartite graphs
#nx_diff         : largest difference between numpy SC and networkx subgraph_centrality (checks the classical reference)
#MAE             : mean absolute error between QSVT SC and exact SC over the vertices
#max_rel_err     : largest relative error |QSVT - exact| / exact over the vertices
#rho, tau        : Spearman and Kendall rank correlation between exact and QSVT SC (1 = identical ranking; exact ties are treated as one class); nan if all SC are equal (K5)
#topk, overlap   : fraction of the exact top-k vertices that QSVT also puts in its top-k (k = 3 for n <= 8, else 5)
tab = pd.DataFrame(rows).T
tab = tab[["n", "edges", "qubits", "bipartite", "classes", "deg_even", "deg_odd", "pe_min", "po_min", "nx_diff", "MAE", "max_rel_err", "rho", "tau", "topk", "overlap"]]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(tab.to_string(formatters={"pe_min": "{:.1e}".format, "po_min": "{:.1e}".format, "nx_diff": "{:.1e}".format,
                                "MAE": "{:.1e}".format, "max_rel_err": "{:.1e}".format, "rho": "{:.3f}".format,
                                "tau": "{:.3f}".format, "overlap": "{:.2f}".format}))

                  n edges qubits bipartite classes deg_even deg_odd   pe_min   po_min  nx_diff      MAE max_rel_err    rho    tau topk overlap
Bull              5     5      5     False       3        8       7  7.9e-02  1.2e-05  2.8e-14  7.4e-06     6.1e-06  1.000  1.000    3    1.00
K3,5              8    15      5      True       2       10    None  5.4e-02      NaN  3.2e-12  1.5e-05     2.7e-06  1.000  1.000    3    1.00
K5                5    10      5     False       1       10      11  6.0e-02  2.7e-02  5.6e-12  1.3e-05     1.2e-06    NaN    NaN    3    1.00
Sedgewick-Maze    8    10      5     False       8        8       9  1.9e-02  1.5e-08  1.4e-13  2.5e-05     2.2e-05  1.000  1.000    3    1.00
Florentine       15    20      6     False      15       10       9  5.7e-03  5.8e-10  5.0e-13  9.7e-06     1.0e-05  1.000  1.000    5    1.00
Lollipop L10,20  30    65      7     False      13       18      19  1.2e-07  6.0e-39  3.1e-11  1.5e-05     9.5e-06  1.000  1.000    5    1.00

In [ ]:
# Florentine families: per-family table and finite-shot study at the frozen DELTA.
# Uses the functions and imports of the first cell (degree_for_eps, DELTA, qml, expm, C, nx, np, pd).
G = nx.florentine_families_graph()
nodes = sorted(G.nodes()); n = len(nodes)
q = int(np.ceil(np.log2(n))); N = 2 ** q                      # 15 families -> 16 states (one isolated dummy vertex)
A = nx.to_numpy_array(G, nodelist=nodes)
A_p = np.zeros((N, N)); A_p[:n, :n] = A
beta = 1.0
lam = np.max(np.abs(np.linalg.eigvalsh(A)))
alpha = 1.001 * max(lam, np.sqrt(np.max(np.abs(A @ A).sum(axis=1))))
z = beta * alpha

# degrees from beta, alpha and DELTA only
d_e, c_e = degree_for_eps(z, DELTA / np.cosh(z), 0)
d_o, c_o = degree_for_eps(z, DELTA / np.sinh(z), 1)
print(f"alpha = {alpha:.6f}   degrees: even {d_e}, odd {d_o}")
print(f"tail error: even {tail_error(z, d_e, 0):.3e} (SC bound {np.cosh(z) * tail_error(z, d_e, 0):.2e}), "
      f"odd {tail_error(z, d_o, 1):.3e} (SC bound {np.sinh(z) * tail_error(z, d_o, 1):.2e})")

# the two circuits
At = A_p / alpha
be_wires = range(q + 1); rp = q + 1
BE = qml.BlockEncode(At, wires=be_wires)
dev = qml.device("default.qubit", wires=q + 2)
def make_circuit(c):
    angles = qml.poly_to_angles(C.cheb2poly(c), "QSVT")
    pos = [qml.PCPhase(a,  dim=N, wires=be_wires) for a in angles]
    neg = [qml.PCPhase(-a, dim=N, wires=be_wires) for a in angles]
    @qml.qnode(dev)
    def circ(v):
        qml.BasisState(np.array([int(b) for b in format(v, f"0{q}b")]), wires=range(1, 1 + q))
        qml.Hadamard(wires=rp)
        qml.ctrl(qml.QSVT(BE, pos), control=rp, control_values=[0])
        qml.ctrl(qml.QSVT(BE, neg), control=rp, control_values=[1])
        qml.Hadamard(wires=rp)
        return qml.probs(wires=range(q + 2))
    return circ
def p_success(circ, v):
    bits = [0] + [int(b) for b in format(v, f"0{q}b")] + [0]
    return circ(v)[int("".join(map(str, bits)), 2)]
circ_e, circ_o = make_circuit(c_e), make_circuit(c_o)
p_e = np.array([p_success(circ_e, v) for v in range(n)])
p_o = np.array([p_success(circ_o, v) for v in range(n)])
sc_odd = np.sinh(z) * np.sqrt(p_o)
sc_qsvt = np.cosh(z) * np.sqrt(p_e) + sc_odd

# classical reference, used only to check the circuit output
sc_exact = np.diag(expm(beta * A))
sc_nx = nx.subgraph_centrality(G)
print(f"networkx vs numpy: {max(abs(sc_nx[v] - sc_exact[i]) for i, v in enumerate(nodes)):.1e}")
print(f"max abs error: {np.max(np.abs(sc_qsvt - sc_exact)):.2e}   max rel error: {np.max(np.abs(sc_qsvt - sc_exact) / sc_exact):.2e}")
order = np.argsort(-sc_exact)
print("ranking exact  :", [nodes[i] for i in order])
print("ranking circuit:", [nodes[i] for i in np.argsort(-sc_qsvt)])
print("ranking from the even part alone:", [nodes[i] for i in np.argsort(-np.cosh(z) * np.sqrt(p_e))])

table = pd.DataFrame({"family": [nodes[i] for i in order],
                      "degree": [G.degree(nodes[i]) for i in order],
                      "exact SC": sc_exact[order],
                      "odd share": (sc_odd / sc_qsvt)[order],
                      "p_even": p_e[order], "p_odd": p_o[order],
                      "SC from circuits": sc_qsvt[order]})
print()
print(table.to_string(index=False, formatters={"exact SC": "{:.5f}".format, "odd share": "{:.4f}".format, "p_even": "{:.4f}".format,
                                               "p_odd": "{:.3e}".format, "SC from circuits": "{:.5f}".format}))

# finite shots: binomial sampling of both circuits, n shots each
srt = np.sort(sc_exact)[::-1]; gaps = (srt[:-1] - srt[1:]) / srt[1:]
j = int(np.argmin(gaps)); i1, i2 = order[j], order[j + 1]
print(f"\nclosest pair: {nodes[i1]} / {nodes[i2]}, {100 * gaps[j]:.2f}% apart")
rng = np.random.default_rng(1)
trials = 2000
print(f"{'shots/circuit':>13} | {'full ranking ok':>15} | {nodes[i1] + '>' + nodes[i2] + ' ok':>22} | {'mean abs SC err':>15}")
for shots in [1000, 10_000, 100_000, 1_000_000, 10_000_000]:
    ce = rng.binomial(shots, p_e, size=(trials, n)); co = rng.binomial(shots, p_o, size=(trials, n))
    est = np.cosh(z) * np.sqrt(ce / shots) + np.sinh(z) * np.sqrt(co / shots)
    rank_ok = np.mean([np.array_equal(np.argsort(-e), order) for e in est])
    pair_ok = np.mean(est[:, i1] > est[:, i2])
    print(f"{shots:>13} | {rank_ok:>15.3f} | {pair_ok:>22.3f} | {np.mean(np.abs(est - sc_exact)):>15.4f}")

alpha = 3.745399   degrees: even 10, odd 9
tail error: even 4.883e-07 (SC bound 1.03e-05), odd 3.211e-06 (SC bound 6.79e-05)
networkx vs numpy: 5.0e-13
max abs error: 2.77e-05   max rel error: 1.00e-05
ranking exact  : ['Medici', 'Strozzi', 'Peruzzi', 'Guadagni', 'Tornabuoni', 'Ridolfi', 'Bischeri', 'Castellani', 'Albizzi', 'Barbadori', 'Salviati', 'Acciaiuoli', 'Lamberteschi', 'Ginori', 'Pazzi']
ranking circuit: ['Medici', 'Strozzi', 'Peruzzi', 'Guadagni', 'Tornabuoni', 'Ridolfi', 'Bischeri', 'Castellani', 'Albizzi', 'Barbadori', 'Salviati', 'Acciaiuoli', 'Lamberteschi', 'Ginori', 'Pazzi']
ranking from the even part alone: ['Medici', 'Strozzi', 'Guadagni', 'Tornabuoni', 'Ridolfi', 'Albizzi', 'Bischeri', 'Peruzzi', 'Castellani', 'Barbadori', 'Salviati', 'Acciaiuoli', 'Lamberteschi', 'Ginori', 'Pazzi']

      family  degree exact SC odd share p_even     p_odd SC from circuits
      Medici       6  7.27522    0.0997 0.0957 1.176e-03          7.27520
     Strozzi       4  5.63206    0.214